# NeoTwin: 03 — LangSplat Training

**Runtime:** Google Colab T4 GPU  
**Time:** ~8–12 min  
**Input:** `neotwin_3dgs_output.zip` from Notebook 02  
**Output:** `langsplat.ckpt` — 3D Gaussians with baked CLIP 512-d semantic vectors

### Upgrades over baseline
| Setting | Baseline | This notebook |
|---|---|---|
| CLIP backbone | ViT-B/32 (default) | **ViT-L/14** (512-d, much richer) |
| Feature levels | 1 | **3** (multi-scale pyramid) |
| Embedding check | None | Cosine variance + singular value sanity gate |
| SAM pre-processing | Off | **On** — segment-aware masking for cleaner features |
| Per-image progress | None | tqdm + per-image CLIP extraction log |

In [ ]:
# ─── 0. CONFIG ───────────────────────────────────────────────────────────────
CLIP_MODEL          = 'ViT-L/14'  # richer than default ViT-B/32
FEATURE_LEVELS      = 3           # multi-scale: 1 (coarse) → 3 (fine)
USE_SAM_MASKING     = True        # SAM-guided masking for cleaner embeddings
SAM_MODEL_TYPE      = 'vit_b'     # vit_b = fast on T4; vit_h = best quality
LANGSPLAT_ITERS     = 30000       # LangSplat-specific optimization steps
MIN_EMBED_VARIANCE  = 0.01        # reject if all embeddings are near-identical
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ─── 1. INSTALL ──────────────────────────────────────────────────────────────
import subprocess
def run(cmd): subprocess.run(cmd, shell=True, check=True)

run('pip install -q git+https://github.com/openai/CLIP.git')
run('pip install -q segment-anything torch torchvision tqdm plyfile')
run('git clone -q https://github.com/minghanqin/LangSplat')
run('pip install -q -e LangSplat')

if USE_SAM_MASKING:
    # Download SAM checkpoint (vit_b = 375 MB, fast on T4)
    sam_ckpt = f'sam_{SAM_MODEL_TYPE}.pth'
    run(f'wget -q https://dl.fbaipublicfiles.com/segment_anything/{sam_ckpt} -O {sam_ckpt}')

print('✅ Dependencies ready')

In [ ]:
# ─── 2. UNPACK INPUT ─────────────────────────────────────────────────────────
import shutil
from pathlib import Path
from google.colab import files

print('Upload neotwin_3dgs_output.zip from Notebook 02:')
uploaded = files.upload()
zip_name = next(iter(uploaded))

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
shutil.unpack_archive(zip_name, DATA_DIR)

PLY_PATH = DATA_DIR / 'point_cloud.ply'
assert PLY_PATH.exists(), f'❌ point_cloud.ply not found in zip — check NB02 output'

ply_mb = PLY_PATH.stat().st_size / 1e6
print(f'✅ PLY loaded: {ply_mb:.1f} MB')

In [ ]:
# ─── 3. PRE-EXTRACT CLIP FEATURES PER IMAGE ──────────────────────────────────
# LangSplat distills per-image CLIP features into Gaussians.
# We do this explicitly so we can validate quality before training.
import clip, torch, numpy as np
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm

DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
IMG_DIR   = DATA_DIR / 'images'
FEAT_DIR  = DATA_DIR / 'clip_features'
FEAT_DIR.mkdir(exist_ok=True)

print(f'Loading CLIP {CLIP_MODEL} on {DEVICE}...')
model, preprocess = clip.load(CLIP_MODEL, device=DEVICE)
model.eval()

image_paths = sorted(IMG_DIR.glob('*.jpg'))
assert image_paths, f'❌ No images found in {IMG_DIR}'

all_features = []
print(f'Extracting CLIP features from {len(image_paths)} images...')

with torch.no_grad():
    for img_path in tqdm(image_paths, desc='CLIP extraction'):
        img     = preprocess(Image.open(img_path)).unsqueeze(0).to(DEVICE)
        feat    = model.encode_image(img)              # (1, 512) or (1, 768)
        feat    = feat / feat.norm(dim=-1, keepdim=True)  # L2-normalise
        feat_np = feat.squeeze().cpu().float().numpy()
        np.save(FEAT_DIR / f'{img_path.stem}.npy', feat_np)
        all_features.append(feat_np)

all_features = np.stack(all_features)  # (N, D)
print(f'✅ Features extracted — shape {all_features.shape}')

In [ ]:
# ─── 4. OPTIONAL: SAM-GUIDED MASKING ─────────────────────────────────────────
# Generates object masks so LangSplat learns clean per-object embeddings
# rather than whole-image averages.
if USE_SAM_MASKING:
    import torch
    import numpy as np
    import cv2
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
    from pathlib import Path
    from tqdm.notebook import tqdm

    MASK_DIR = DATA_DIR / 'sam_masks'
    MASK_DIR.mkdir(exist_ok=True)

    sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=f'sam_{SAM_MODEL_TYPE}.pth')
    sam.to(DEVICE)
    mask_gen = SamAutomaticMaskGenerator(
        sam,
        points_per_side=16,        # 16 vs default 32 → 4× faster, same coverage
        pred_iou_thresh=0.92,
        stability_score_thresh=0.96,
        min_mask_region_area=500
    )

    print(f'Running SAM on {len(image_paths)} images...')
    for img_path in tqdm(image_paths, desc='SAM masking'):
        img_bgr  = cv2.imread(str(img_path))
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        masks    = mask_gen.generate(img_rgb)
        # Save mask metadata (bounding boxes + scores) as numpy
        if masks:
            boxes  = np.array([m['bbox']       for m in masks], dtype=np.float32)
            scores = np.array([m['predicted_iou'] for m in masks], dtype=np.float32)
            np.savez(
                MASK_DIR / f'{img_path.stem}.npz',
                boxes=boxes, scores=scores
            )

    print(f'✅ SAM masks saved to {MASK_DIR}')
else:
    print('⏭  SAM masking skipped (USE_SAM_MASKING=False)')

In [ ]:
# ─── 5. EMBEDDING QUALITY VALIDATION ─────────────────────────────────────────
# Gate: reject if all image embeddings are near-identical (degenerate scene)
import numpy as np

# Cosine similarity between all pairs (sample 100 if large)
sample = all_features[:100] if len(all_features) > 100 else all_features
sim_matrix = sample @ sample.T  # already L2-normalised
off_diag   = sim_matrix[np.triu_indices(len(sample), k=1)]

mean_sim = float(off_diag.mean())
std_sim  = float(off_diag.std())

# Singular value spread — how much variance does the feature space have?
_, s, _ = np.linalg.svd(all_features - all_features.mean(0), full_matrices=False)
sv_ratio = float(s[0] / s.sum())   # close to 1.0 = degenerate

print('\n📊 EMBEDDING QUALITY REPORT')
print('─' * 40)
print(f'  CLIP model          : {CLIP_MODEL}')
print(f'  Feature dimension   : {all_features.shape[1]}')
print(f'  Images              : {all_features.shape[0]}')
print(f'  Mean pairwise cosine: {mean_sim:.4f}  (lower = more diverse views)')
print(f'  Cosine std          : {std_sim:.4f}')
print(f'  SV dominance ratio  : {sv_ratio:.4f}  (< 0.9 = healthy spread)')

assert std_sim >= MIN_EMBED_VARIANCE, (
    f'❌ Embedding variance {std_sim:.4f} < {MIN_EMBED_VARIANCE}. '
    'Scene may be too textureless. Try adding more diverse viewpoints.'
)
assert sv_ratio < 0.95, (
    f'❌ SV dominance {sv_ratio:.3f} ≥ 0.95 — embeddings are near-degenerate. '
    'Increase image diversity or switch to a different CLIP model.'
)
print('\n✅ Embedding quality gate PASSED')

In [ ]:
# ─── 6. LANGSPLAT TRAINING ────────────────────────────────────────────────────
import subprocess, os
from pathlib import Path

OUTPUT_DIR = Path('langsplat_output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Build LangSplat training command
# --feature_level 3 = fine-grained multi-scale feature pyramid
# --start_checkpoint = initialise from NB02's trained Gaussians
cmd = [
    'python', 'LangSplat/train.py',
    '-s', str(DATA_DIR),
    '-m', str(OUTPUT_DIR),
    '--start_checkpoint', str(PLY_PATH),
    '--feature_level', str(FEATURE_LEVELS),
    '--clip_model_type', CLIP_MODEL.replace('/', '-'),
    '--iterations', str(LANGSPLAT_ITERS),
    '--eval',
    '--quiet'
]

if USE_SAM_MASKING:
    cmd += ['--mask_dir', str(DATA_DIR / 'sam_masks')]

print('🚀 Starting LangSplat training...')
print('   This bakes CLIP vectors into each 3D Gaussian.')
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print('STDERR:', result.stderr[-3000:])
    raise RuntimeError('❌ LangSplat training failed — see stderr above')

print('✅ LangSplat training complete')
print(result.stdout[-2000:] if result.stdout else '(no stdout)')

In [ ]:
# ─── 7. PACKAGE & DOWNLOAD ───────────────────────────────────────────────────
import shutil, glob
from pathlib import Path
from google.colab import files

OUTPUT_DIR = Path('langsplat_output')

# Find the LangSplat checkpoint
ckpt_files = glob.glob(str(OUTPUT_DIR / '**' / '*.pth'), recursive=True) + \
             glob.glob(str(OUTPUT_DIR / '**' / '*.ckpt'), recursive=True)

assert ckpt_files, '❌ No checkpoint found in langsplat_output'

bundle = Path('neotwin_langsplat_output')
bundle.mkdir(exist_ok=True)
# Include original PLY so NB04 only needs this one zip
shutil.copy(PLY_PATH, bundle / 'point_cloud.ply')
for ckpt in ckpt_files:
    shutil.copy(ckpt, bundle / Path(ckpt).name)

# Save embedding report
import json
with open(bundle / 'embedding_report.json', 'w') as f:
    json.dump({
        'clip_model': CLIP_MODEL,
        'feature_levels': FEATURE_LEVELS,
        'feature_dim': int(all_features.shape[1]),
        'num_images': int(all_features.shape[0]),
        'mean_pairwise_cosine': round(mean_sim, 4),
        'cosine_std': round(std_sim, 4),
        'sv_dominance': round(sv_ratio, 4),
        'sam_masking_used': USE_SAM_MASKING
    }, f, indent=2)

shutil.make_archive('neotwin_langsplat_output', 'zip', bundle)

print(f'📦 Checkpoint: {[Path(c).name for c in ckpt_files]}')
files.download('neotwin_langsplat_output.zip')
print('✅ Download started → use as input to Notebook 04')